# Warm start on the IDAES CSTR

Under closed-loop control the next problem is the last one moved one
step, so the last solution moved one step is nearly its answer.
`drto.warm_start_dynamic` shifts it. This notebook runs one loop
iteration on the pattern a loop actually uses: one persistent model,
built and scaled once, then solved, shifted, and solved again, with
nothing passed to the second solve beyond the scaling. The shift
carries values only; the solver rebuilds its own multipliers from a
good starting point in its first iterations, faster than any
multipliers or barrier settings we could hand it. The solver is
pounce, which reads the scaling suffix through its standard
interface.

## One model, built and scaled once

Declarations, the setpoint, the terminal segment, the cold start with
the flowsheet's magnitudes as its `scale` source, energy in joules near
`1e7` and duties in watts near `1e6`, then the assembly and
`drto.scale` writing the same magnitudes for the solves. The whole
loop keeps this one model, and every solve receives the factors under
`nlp_scaling_method=user-scaling`.

In [1]:
import contextlib, io, time

import pyomo.environ as pyo

import drto
from models.idaes_cstr import DC_START, F_IN, VOLUME, build

m = build()

ss = pyo.TransformationFactory("drto.steady_state_simulation").create_using(
    m, controls={m.fs.cstr.control_volume.heat.name: 0.0,
                 m.fs.cstr.inlet.flow_vol.name: F_IN})
drto.initialize_steady_state(ss)
drto.scaled_solve(ss)
cvs = ss.fs.cstr.control_volume
for j, ssp in (("NaOH", m.ss_naoh), ("EthylAcetate", m.ss_ea),
               ("SodiumAcetate", m.ss_sa), ("Ethanol", m.ss_etoh)):
    ssp.set_value(pyo.value(cvs.material_holdup["Liq", j]))
for j, sgn in (("NaOH", 1), ("EthylAcetate", 1),
               ("SodiumAcetate", -1), ("Ethanol", -1)):
    m.mat0[j] = pyo.value(cvs.material_holdup["Liq", j]) + sgn * DC_START * VOLUME
for k in m.eng_ss:
    m.eng_ss[k] = pyo.value(cvs.energy_holdup[k])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
drto.cold_start_dynamic(m, profile="exponential", time_constant=3.0,
                        scale={"J": 1e7, "W": 1e6})
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)

drto.scale(m, source={"J": 1e7, "W": 1e6})
res = pyo.SolverFactory("pounce_v2").solve(
    m, options={"nlp_scaling_method": "user-scaling"}, tee=True)
print(res.solver.termination_condition)

component keys that are not exported as part of the NL file.  Skipping.


component keys that are not exported as part of the NL file.  Skipping.


component keys that are not exported as part of the NL file.  Skipping.


********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.10.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4472
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      645

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  6.7507105e+03 1.59e+07 1.02e+03   -1.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1 -1.5205667e+03 5.56e+07 9.98e+02   -1.0 1.91e+06      - 1.4

   9  2.3508797e+03 1.87e+07 1.20e+02   -1.0 4.04e+05      - 7.43e-01 1.00e+00h  1
iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
  10  3.5165505e+03 3.87e+05 3.59e+00   -1.0 1.37e+06      - 9.90e-01 1.00e+00h  1
  11  3.5384911e+03 5.25e+03 1.66e-02   -1.0 1.35e+08      - 9.91e-01 1.00e+00h  1
  12  3.5386484e+03 1.72e-01 2.16e-06   -1.0 3.95e+09      - 1.00e+00 1.00e+00f  1
  13  3.5376444e+03 2.78e+01 2.28e-03   -2.5 4.11e+07      - 1.00e+00 1.00e+00f  1
  14  3.5376328e+03 2.56e-02 2.56e-06   -3.8 7.51e+07      - 1.00e+00 1.00e+00h  1
  15  3.5376313e+03 1.95e-03 2.14e-08   -5.7 1.93e-03   -5.0 1.00e+00 1.00e+00h  1
  16  3.5376313e+03 6.84e-03 8.10e-08   -8.6 1.38e+06      - 1.00e+00 1.00e+00h  1
  17  3.5376313e+03 4.88e-03 3.23e-12   -8.6 7.46e-08   -5.4 1.00e+00 1.00e+00h  1


Number of Iterations....: 17

                                   (scaled)                 (unscaled)
Objective...............:   3.5376313199950446e+03    3.53763131999

optimal


## One step later: shift everything

The loop implements the first move and the state advances one sample;
the model's own solution at t = h stands in for the measurement, read
directly, since the model never leaves its own units. The shift moves
every variable one sampling time forward.

In [2]:
cv = m.fs.cstr.control_volume
h = 1.0
for j in ("NaOH", "EthylAcetate", "SodiumAcetate", "Ethanol"):
    m.mat0[j] = pyo.value(cv.material_holdup[h, "Liq", j])
m.eng0["Liq"] = pyo.value(cv.energy_holdup[h, "Liq"])

print(drto.warm_start_dynamic(m))

drto warm_start_dynamic (the previous solution, one step on)
  shift         : 1 time units
  copied        : 1009 values on aligned points
  interpolated  : 496 values between points
  filled        : 0 values past the end
  tail          : shifted through t = tN + atanh(tau)/gamma


## The second solve, warm

The same model again from the shifted values, with the same scaling and
nothing else: the solver starts at the point the shift left, rebuilds
its multipliers from it, and stops.

In [3]:
res = pyo.SolverFactory("pounce_v2").solve(
    m, options={"nlp_scaling_method": "user-scaling"}, tee=True)
print(res.solver.termination_condition)

component keys that are not exported as part of the NL file.  Skipping.


********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.10.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4472
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      645

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  1.6659160e+02 1.71e+07 1.02e+03   -1.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1  6.6539174e+01 1.68e+07 9.54e+02   -1.0 2.23e+04      - 7.9

   8  6.5591620e+01 7.32e-04 1.60e-07   -5.7 1.60e-03   -4.0 1.00e+00 1.00e+00h  1
   9  6.5591602e+01 1.34e-03 7.57e-07   -8.6 9.62e+05      - 1.00e+00 1.00e+00h  1
iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
  10  6.5591602e+01 1.65e-03 9.23e-12   -8.6 1.49e-08   -4.5 1.00e+00 1.00e+00h  1


Number of Iterations....: 10

                                   (scaled)                 (unscaled)
Objective...............:   6.5591601673382982e+01    6.5591601673382982e+01
Dual infeasibility......:   9.2250101871652057e-12    9.2250101871652057e-12
Constraint violation....:   1.1641532182693481e-10    1.6479492187500000e-03
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   2.5059035596801201e-09    2.5059035596801201e-09
Overall NLP error.......:   2.5059035596801201e-09    1.6479492187500000e-03


Number of objective function evaluations             = 11
Number of objective gradient evaluations   

optimal


The cold solve above took seventeen iterations and the warm-started one
takes ten: the shifted solution is nearly the answer, and the solve
rebuilds its multipliers from it and stops. That is warm starting a
receding horizon, whole.